# Docker Basics Worksheet
**NAVTTC / NIAI AI/ML Trainer Cohort**. Runs in Colab or Jupyter, plain Python. **Tasks 1 to 5 need no Docker.** Fill every `None` / `# YOUR CODE` area, then run top to bottom. Check cells print `OK` or `FIX`.

Running example: the fraud detection model from the MLOps lecture.

In [ ]:
def check(name, ok, hint=""):
    print(("OK   " if ok else "FIX  ") + name + ("" if ok else "  -> " + hint))
print("ready")

## Task 1: Image is a class, container is an object

Finish the `Image` class. `run()` should return a new `Container` each time. A container has a `name` (from the image) and its own list `files`, which starts empty. Then show that writing a file in one container does not change another.

In [ ]:
class Container:
    def __init__(self, image_name):
        self.name = image_name
        self.files = []          # each container has its OWN files

class Image:
    def __init__(self, name):
        self.name = name
    def run(self):
        return None              # YOUR CODE: return a new Container

In [ ]:
img = Image("fraud-model:1.0")
c1, c2 = img.run(), img.run()
c1.files.append("log.txt")
check("two different containers", c1 is not c2, "run() must create a new Container each time")
check("same image name", c1.name == c2.name == "fraud-model:1.0", "")
check("containers do not share files", c1.files == ["log.txt"] and c2.files == [], "files must be separate per container")

## Task 2: Why containers are light (example numbers)

One machine has **8 GB** of memory. Made-up numbers for the idea: every **VM** needs 2.0 GB for its own operating system plus 1.0 GB for the app. A **container** needs about 0.1 GB of overhead plus 1.0 GB for the app (the kernel is shared). Compute the total for 3 projects each way and whether it fits in 8 GB.

In [ ]:
HOST_GB = 8
APP_GB = 1.0
vm_total = None            # YOUR CODE: 3 VMs
container_total = None     # YOUR CODE: 3 containers
vm_fits = None             # True/False
container_fits = None

In [ ]:
print(f"3 VMs: {vm_total} GB (fits: {vm_fits}) | 3 containers: {container_total:.1f} GB (fits: {container_fits})")
check("VM total", vm_total == 9.0, "3 x (2.0 + 1.0)")
check("container total", abs(container_total - 3.3) < 1e-9, "3 x (0.1 + 1.0)")
check("conclusion", vm_fits is False and container_fits is True, "9 GB does not fit in 8 GB; 3.3 GB does")
print("These numbers are an illustration, not measurements. Real overhead depends on the setup.")

## Task 3: Put the Dockerfile in order

These lines are mixed up. Write them in the right order so that the image builds (hint: you cannot install libraries before the base image, and copying `requirements.txt` first lets Docker reuse its cache).

In [ ]:
lines = [
    'CMD ["python", "app.py"]',
    'COPY . .',
    'FROM python:3.11-slim',
    'RUN pip install -r requirements.txt',
    'WORKDIR /app',
    'COPY requirements.txt .',
]
ordered = []        # YOUR CODE: the same six lines in the right order

In [ ]:
def pos(prefix): 
    return next((i for i, l in enumerate(ordered) if l.startswith(prefix)), None)
check("same six lines", sorted(ordered) == sorted(lines), "use every line once")
check("FROM first", bool(ordered) and ordered[0].startswith("FROM"), "")
check("requirements before install", pos("COPY requirements") is not None and pos("COPY requirements") < pos("RUN pip"), "")
check("install before copying code", pos("RUN pip") is not None and pos("RUN pip") < pos("COPY . ."), "that is what makes the cache work")
check("CMD last", bool(ordered) and ordered[-1].startswith("CMD"), "")

## Task 4: Pin the library versions

Reproducibility means exact versions. Write `pin(names)` that returns a list like `numpy==2.1.0` using the versions **installed in this notebook** (`importlib.metadata.version`).

In [ ]:
from importlib.metadata import version
def pin(names):
    return None            # YOUR CODE: [f"{n}=={version(n)}" for n in names]

In [ ]:
reqs = pin(["numpy", "pandas"])
print("\n".join(reqs))
check("two lines", isinstance(reqs, list) and len(reqs) == 2, "")
check("pinned format", all("==" in r for r in reqs), "use name==version")
check("matches installed", reqs[0] == f"numpy=={version('numpy')}", "")
print("Your versions will differ from your neighbour's. That is exactly the problem Docker solves.")

## Task 5: Write the files for the fraud container

Write three files into a folder `fraud_container/`. `app.py` should define `risk_score(amount, usual_amount, new_country=False)` (the function from the CI/CD worksheet) and print the score for a sample payment. The Dockerfile should be your ordered list from Task 3. `requirements.txt` can be empty or hold your pins.

In [ ]:
import os
os.makedirs("fraud_container", exist_ok=True)
app_py = ""            # YOUR CODE: a string with the Python code
open("fraud_container/app.py", "w").write(app_py)
open("fraud_container/Dockerfile", "w").write("\n".join(ordered) + "\n")
open("fraud_container/requirements.txt", "w").write("")

In [ ]:
import subprocess, sys
ok_files = all(os.path.exists(f"fraud_container/{f}") for f in ["app.py", "Dockerfile", "requirements.txt"])
check("three files exist", ok_files, "")
out = subprocess.run([sys.executable, "fraud_container/app.py"], capture_output=True, text=True)
print(out.stdout.strip() or out.stderr.strip())
check("app.py runs and prints a risk", out.returncode == 0 and "risk" in out.stdout, "app.py must print a line with 'risk'")
check("Dockerfile starts with FROM", open("fraud_container/Dockerfile").read().startswith("FROM"), "")

## Part B (optional, needs Docker installed)
Run these in a terminal. Check the output on your own machine.
```
docker pull hello-world
docker run hello-world
cd fraud_container
docker build -t fraud-model:1.0 .
docker run fraud-model:1.0
docker images
docker ps -a
```
**Check:** the last `docker run` should print a line like `risk: 100`. If not, read the error from the bottom up.

**Reflection (2 sentences each):** Why is `fraud-model:1.0` better than 'it works on my laptop'? When would you choose a VM instead of a container?